# MIS 433 Final Project: AI Investment Signals

For this project, we are looking at a few major AI-related companies and seeing if stock trends, volume, volatility, and news sentiment can help give a short-term investment signal.

The companies we are using are NVDA, MSFT, GOOGL, AMZN, AMD, and AVGO. Instead of trying to predict the exact stock price, we are focusing on whether the stock might move up or down over the next 7 trading days.


## 1. Environment Setup

This section sets up the notebook. It works in Colab, but it can also run locally in VS Code from the project folder.


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/MIS_433_Project')
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_DIR)
    print(f"Running in Google Colab: {PROJECT_DIR}")
except ImportError:
    current_dir = Path.cwd().resolve()
    PROJECT_DIR = current_dir.parent if current_dir.name == 'notebooks' else current_dir
    os.chdir(PROJECT_DIR)
    print(f"Running locally in VS Code: {PROJECT_DIR}")

Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('data/external').mkdir(parents=True, exist_ok=True)
Path('outputs/charts').mkdir(parents=True, exist_ok=True)
Path('outputs/model_results').mkdir(parents=True, exist_ok=True)
Path('outputs/screenshots').mkdir(parents=True, exist_ok=True)


In [ ]:
!pip install yfinance pandas numpy matplotlib seaborn scikit-learn requests google-generativeai


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_theme(style="whitegrid")


## 2. Stock Data Collection

Here we use `yfinance` to pull the historical stock data from Yahoo Finance for each company.


In [ ]:
tickers = ["NVDA", "MSFT", "GOOGL", "AMZN", "AMD", "AVGO"]
start_date = "2020-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")


In [ ]:
stock_data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    group_by="ticker"
)

stock_data.head()


In [ ]:
stock_data.to_csv("data/raw/stock_prices_raw.csv")


## 3. Data Cleaning

The stock data comes in a wide format, so this section cleans it up into one table that is easier to use.


In [ ]:
clean_data = []

for ticker in tickers:
    temp = stock_data[ticker].copy()
    temp["ticker"] = ticker
    temp = temp.reset_index()
    clean_data.append(temp)

stock_df = pd.concat(clean_data, ignore_index=True)
stock_df.columns = [col.lower().replace(" ", "_") for col in stock_df.columns]
stock_df["date"] = pd.to_datetime(stock_df["date"])

stock_df.head()


In [ ]:
stock_df.to_csv("data/processed/stock_prices_clean.csv", index=False)


## 4. Exploratory Data Analysis

This section looks at the data with a few basic charts and summary stats so we can compare the companies.


In [ ]:
plt.figure(figsize=(12, 6))

for ticker in tickers:
    temp = stock_df[stock_df["ticker"] == ticker]
    plt.plot(temp["date"], temp["close"], label=ticker)

plt.title("Historical Closing Prices for AI Infrastructure Stocks")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
stock_df.info()
stock_df.describe()


## 5. Feature Engineering

Here we add new columns that can help with the model, like returns, moving averages, volatility, and volume change.


In [ ]:
stock_df = stock_df.sort_values(["ticker", "date"])

stock_df["daily_return"] = stock_df.groupby("ticker")["close"].pct_change()
stock_df["return_7d"] = stock_df.groupby("ticker")["close"].pct_change(7)
stock_df["return_30d"] = stock_df.groupby("ticker")["close"].pct_change(30)

stock_df["ma_7d"] = stock_df.groupby("ticker")["close"].transform(lambda x: x.rolling(window=7).mean())
stock_df["ma_30d"] = stock_df.groupby("ticker")["close"].transform(lambda x: x.rolling(window=30).mean())
stock_df["ma_90d"] = stock_df.groupby("ticker")["close"].transform(lambda x: x.rolling(window=90).mean())

stock_df["volatility_30d"] = stock_df.groupby("ticker")["daily_return"].transform(lambda x: x.rolling(window=30).std())
stock_df["volume_change"] = stock_df.groupby("ticker")["volume"].pct_change()

stock_df.head()


In [ ]:
stock_df.to_csv("data/processed/stock_features.csv", index=False)


## 6. Stock Performance Comparisons

This section compares the stocks visually. The normalized chart helps because the companies all have different stock prices.


In [ ]:
normalized_df = stock_df.copy()
normalized_df["normalized_close"] = normalized_df.groupby("ticker")["close"].transform(lambda x: x / x.iloc[0] * 100)

plt.figure(figsize=(12, 6))

for ticker in tickers:
    temp = normalized_df[normalized_df["ticker"] == ticker]
    plt.plot(temp["date"], temp["normalized_close"], label=ticker)

plt.title("Normalized Stock Performance Since 2020")
plt.xlabel("Date")
plt.ylabel("Indexed Price: 2020 = 100")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/charts/normalized_stock_performance.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
volatility_summary = stock_df.groupby("ticker")["daily_return"].std().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
volatility_summary.plot(kind="bar")
plt.title("Volatility Comparison by Stock")
plt.xlabel("Ticker")
plt.ylabel("Standard Deviation of Daily Returns")
plt.tight_layout()
plt.savefig("outputs/charts/volatility_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
summary = stock_df.groupby("ticker").agg(
    start_price=("close", "first"),
    end_price=("close", "last"),
    avg_daily_return=("daily_return", "mean"),
    volatility=("daily_return", "std"),
    avg_volume=("volume", "mean")
).reset_index()

summary["total_return"] = (summary["end_price"] / summary["start_price"]) - 1
summary = summary.sort_values("total_return", ascending=False)

summary


## 7. News Sentiment Data from Alpha Vantage

This section pulls recent news sentiment from Alpha Vantage. The sentiment score is positive for more positive news, negative for more negative news, and close to zero for neutral news.

The API key can come from Colab Secrets or from a local `.env` file. This keeps the key out of the notebook and out of GitHub.


In [ ]:
import os
import requests
import time

try:
    from google.colab import userdata
    alpha_key = userdata.get("ALPHA_VANTAGE_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    alpha_key = os.getenv("ALPHA_VANTAGE_API_KEY")

if not alpha_key or alpha_key == "your_alpha_vantage_api_key_here":
    raise ValueError("Alpha Vantage API key not found. Add it to Colab Secrets or your local .env file.")

all_sentiment = []

for ticker in tickers:
    print(f"Pulling sentiment for {ticker}...")

    params = {
        "function": "NEWS_SENTIMENT",
        "tickers": ticker,
        "apikey": alpha_key,
        "limit": 50
    }

    response = requests.get("https://www.alphavantage.co/query", params=params)
    data = response.json()

    if "feed" not in data:
        print(f"No feed returned for {ticker}: {data}")
        continue

    for article in data.get("feed", []):
        for ticker_info in article.get("ticker_sentiment", []):
            if ticker_info["ticker"] == ticker:
                all_sentiment.append({
                    "ticker": ticker,
                    "date": article["time_published"][:8],
                    "title": article["title"],
                    "source": article["source"],
                    "url": article["url"],
                    "ticker_sentiment_score": float(ticker_info["ticker_sentiment_score"]),
                    "ticker_sentiment_label": ticker_info["ticker_sentiment_label"],
                    "relevance_score": float(ticker_info["relevance_score"])
                })

    time.sleep(12)

sentiment_df = pd.DataFrame(all_sentiment)
sentiment_df.head()


In [ ]:
sentiment_df["date"] = pd.to_datetime(sentiment_df["date"], format="%Y%m%d")

daily_sentiment = sentiment_df.groupby(["ticker", "date"]).agg(
    avg_sentiment_score=("ticker_sentiment_score", "mean"),
    avg_relevance_score=("relevance_score", "mean"),
    article_count=("title", "count")
).reset_index()

daily_sentiment


In [ ]:
sentiment_df.to_csv("data/external/alpha_vantage_news_sentiment_raw.csv", index=False)
daily_sentiment.to_csv("data/external/daily_sentiment_scores.csv", index=False)

daily_sentiment["ticker"].value_counts()


## 8. Combining Stock and Sentiment Data

Here we add the latest sentiment score for each company to the stock dataset so it can be used as another feature.


In [ ]:
latest_sentiment = daily_sentiment.sort_values("date").groupby("ticker").tail(1)

latest_sentiment = latest_sentiment[[
    "ticker",
    "avg_sentiment_score",
    "avg_relevance_score",
    "article_count"
]]

stock_df = pd.read_csv("data/processed/stock_features.csv")
stock_df["date"] = pd.to_datetime(stock_df["date"])

stock_df = stock_df.merge(latest_sentiment, on="ticker", how="left")

stock_df["avg_sentiment_score"] = stock_df["avg_sentiment_score"].fillna(0)
stock_df["avg_relevance_score"] = stock_df["avg_relevance_score"].fillna(0)
stock_df["article_count"] = stock_df["article_count"].fillna(0)

stock_df.head()


In [ ]:
stock_df.to_csv("data/processed/stock_features_with_sentiment.csv", index=False)


In [ ]:
sentiment_summary = latest_sentiment.sort_values("avg_sentiment_score", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(sentiment_summary["ticker"], sentiment_summary["avg_sentiment_score"])
plt.title("Latest Average News Sentiment by Company")
plt.xlabel("Ticker")
plt.ylabel("Average Sentiment Score")
plt.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.savefig("outputs/charts/latest_sentiment_by_company.png", dpi=300, bbox_inches="tight")
plt.show()


## 9. Prediction Target

This section creates the target column for the model. A `1` means the stock went up 7 trading days later, and a `0` means it did not.


In [ ]:
stock_df = pd.read_csv("data/processed/stock_features_with_sentiment.csv")
stock_df["date"] = pd.to_datetime(stock_df["date"])
stock_df = stock_df.sort_values(["ticker", "date"])

stock_df["future_close_7d"] = stock_df.groupby("ticker")["close"].shift(-7)
stock_df["future_return_7d"] = (stock_df["future_close_7d"] / stock_df["close"]) - 1
stock_df["target_up_7d"] = (stock_df["future_return_7d"] > 0).astype(int)

stock_df[[
    "date", "ticker", "close", "future_close_7d",
    "future_return_7d", "target_up_7d"
]].head(10)


In [ ]:
stock_df.to_csv("data/processed/model_ready_stock_data.csv", index=False)


## 10. Next Steps

The next step is to build and test a simple model that predicts whether each stock goes up or down over the next 7 trading days.
